# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset is provided via a Croissant schema and contains several record sets, fields, and columns designed for in-depth data science workflows.

### Dataset Source
We use the Croissant schema directly from:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata (not as a dict, but as an object)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields (columns) and their IDs.

Record sets, fields, and columns are referenced via their `@id` for unambiguous identification.

In [ ]:
# List all record sets and their fields
print("Available record sets and their fields:\n")

for rs in dataset.metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    print("Fields (as columns):")
    for field in rs.fields:
        print(f"  - Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()
# Example: print a single example record from the first record set
if len(dataset.metadata.record_sets) > 0:
    first_rs_id = dataset.metadata.record_sets[0].id
    print(f"First record from record set '{first_rs_id}':")
    for record in dataset.records(record_set=first_rs_id):
        print(record)
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are by the record set and field `@id`s shown above.

We'll load all record sets into separate DataFrames using their `@id`.

In [ ]:
# Extract all record sets into DataFrames, referenced by @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
for rs in dataset.metadata.record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs.name} (@id={rs_id}), fields: {list(df.columns)}\nRows: {len(df)}\n")

# Show columns of first record set
if len(record_set_ids) > 0:
    first_rs = record_set_ids[0]
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data preparation steps—filter, normalize, and group numeric fields. All column references use entity `@id`s from the metadata section.

As an example, we'll select a numeric field from the first record set, filter by a threshold, normalize it, and group by another categorical field if available.

In [ ]:
# Select an example numeric field and group field by @id for EDA
selected_rs_id = record_set_ids[0] if record_set_ids else None
numeric_field_id = None
group_field_id = None

if selected_rs_id:
    # Try to automatically select numeric and group fields
    fields = dataset.metadata.record_sets[0].fields
    for field in fields:
        if field.data_type in ('Float', 'Integer', 'Number') and numeric_field_id is None:
            numeric_field_id = field.id
        if field.data_type == 'Text' and group_field_id is None:
            group_field_id = field.id
    print(f"Numeric field chosen: {numeric_field_id}")
    print(f"Group/categorical field chosen: {group_field_id if group_field_id else 'None'}")

    df = dataframes[selected_rs_id].copy()

    # Some columns are strings; try to convert numeric field to float
    if numeric_field_id and numeric_field_id in df.columns:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group if possible
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} and averaged {numeric_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and grouped means if available.

_Note: The following cell assumes `matplotlib` and `seaborn` are installed. If not, install via `!pip install matplotlib seaborn`._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[selected_rs_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '{selected_rs_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped bar if grouping was possible
    if group_field_id and group_field_id in dataframes[selected_rs_id].columns:
        group_means = dataframes[selected_rs_id].groupby(group_field_id)[numeric_field_id].mean().dropna()
        plt.figure(figsize=(10, 4))
        group_means.plot(kind='bar')
        plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant datasets via `mlcroissant`.
- Enumerate and reference record sets, fields, and columns by their `@id` for robust, schema-driven workflows.
- Import structured tables as DataFrames, filter and transform data using the Croissant metadata (`@id`s).
- Visualize key attributes of the dataset.

**Key findings and next steps:**
- The FAIR<sup>2</sup> example dataset allows systematic proteomics exploration, with all objects schema-linked for full reproducibility.
- The above pattern can be readily extended to machine learning or complex statistical analysis by pivoting on the schema metadata and entity IDs.

_Explore further using the Croissant `metadata` for rich data introspection and downstream FAIR workflows!_